In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, NMF
from gensim.parsing.preprocessing import STOPWORDS
import matplotlib.pyplot as plt

nlp = spacy.load('en_core_web_sm')

GENIUS_TOKEN = os.environ["GENIUS_TOKEN"]

337


In [ ]:
dainos = {
    "pop": {
        "Lorde": [
            "Royals", "Team", "Ribs", "Tennis Court", "Green Light",
            "Liability", "Homemade Dynamite", "Solar Power",
            "Stoned at the Nail Salon", "The Louvre"
        ],
        "Taylor Swift": [
            "Love Story", "Blank Space", "All Too Well",
            "You Belong With Me", "Cardigan", "Willow",
            "Wildest Dreams", "Style", "Anti-Hero", "Cruel Summer"
        ],
        "Billie Eilish": [
            "Bad Guy", "Bury a Friend", "When the Party’s Over",
            "Ocean Eyes", "Everything I Wanted", "Bellyache",
            "Happier Than Ever", "You Should See Me in a Crown",
            "Therefore I Am", "Lovely"
        ],
        "Adele": [
            "Hello", "Someone Like You", "Rolling in the Deep",
            "Set Fire to the Rain", "Easy On Me", "Skyfall",
            "When We Were Young", "Send My Love",
            "Make You Feel My Love", "Turning Tables"
        ],
        "Sia": [
            "Chandelier", "Elastic Heart", "Cheap Thrills", "Alive",
            "The Greatest", "Unstoppable", "Bird Set Free",
            "Big Girls Cry", "Breathe Me", "Soon We'll Be Found"
        ]
    },

    "rap": {
        "50 Cent": [
            "In Da Club", "Candy Shop", "P.I.M.P.", "21 Questions",
            "Many Men", "Ayo Technology", "Just a Lil Bit",
            "Hustler’s Ambition", "Window Shopper", "If I Can’t"
        ],
        "Eminem": [
            "Lose Yourself", "Stan", "The Real Slim Shady", "Without Me",
            "Mockingbird", "Not Afraid", "Rap God", "Sing for the Moment",
            "Love the Way You Lie", "Till I Collapse"
        ],
        "Kendrick Lamar": [
            "HUMBLE.", "DNA.", "Alright", "Bitch, Don't Kill My Vibe",
            "Money Trees", "Swimming Pools", "King Kunta",
            "The Blacker the Berry", "Sing About Me, I'm Dying of Thirst",
            "Poetic Justice"
        ],
        "J. Cole": [
            "No Role Modelz", "Wet Dreamz", "Power Trip", "Love Yourz",
            "Middle Child", "Fire Squad", "G.O.M.D", "Apparently",
            "Ville Mentality", "1985"
        ],
        "NF": [
            "Let You Down", "The Search", "If You Want Love",
            "When I Grow Up", "Lie", "Time", "Paralyzed",
            "Therapy Session", "How Could You Leave Us", "Outro"
        ]
    },

    "rock": {
        "Radiohead": [
            "Creep", "Karma Police", "Paranoid Android", "No Surprises",
            "High and Dry", "Street Spirit (Fade Out)",
            "Everything in Its Right Place", "Pyramid Song",
            "Idioteque", "Fake Plastic Trees"
        ],
        "Arctic Monkeys": [
            "Do I Wanna Know?", "R U Mine?", "505", "Arabella",
            "I Bet You Look Good on the Dancefloor",
            "Why'd You Only Call Me When You're High?",
            "Fluorescent Adolescent", "Snap Out of It",
            "Crying Lightning", "Cornerstone"
        ],
        "Red Hot Chili Peppers": [
            "Californication", "Under the Bridge", "Scar Tissue",
            "Otherside", "By the Way", "Dani California",
            "Snow (Hey Oh)", "Can't Stop", "Give It Away",
            "Soul to Squeeze"
        ],
        "Muse": [
            "Uprising", "Starlight", "Supermassive Black Hole",
            "Hysteria", "Time Is Running Out", "Knights of Cydonia",
            "Madness", "Resistance", "Plug In Baby", "Dead Inside"
        ],
        "The Killers": [
            "Mr. Brightside", "Somebody Told Me", "When You Were Young",
            "All These Things That I've Done", "Human", "Read My Mind",
            "Smile Like You Mean It", "Runaways", "Shot at the Night",
            "Spaceman"
        ]
    }
}

In [359]:
def gauti_zodzius(pavadinimas, atlikejas):
    headers = {"Authorization": f"Bearer {GENIUS_TOKEN}"}
    api_url = f"https://api.genius.com/search?q={pavadinimas} {atlikejas}"
    uzklausa = requests.get(api_url, headers=headers)

    try:
        duomenys = uzklausa.json()
    except:
        print("Duomenys nepasiekiami")
        return None

    dainos_radimas = duomenys["response"].get("hits", [])
    if not dainos_radimas:
        print(f"Nerasta daina")
        return None

    dainu_linkas = dainos_radimas[0]["result"]["path"]
    api_url = "https://genius.com" + dainu_linkas
    puslapis = requests.get(api_url)
    soup = BeautifulSoup(puslapis.text, "html.parser")

    lyrics_divs = soup.find_all("div", {"data-lyrics-container": "true"})
    if not lyrics_divs:
        print("Nerasti žodžiai")
        return None

    lyrics_list = [div.get_text(separator="\n") for div in lyrics_divs]
    lyrics = "\n".join(lyrics_list)
    
    lyrics = re.split(r"Read More|Translations|Contributors", 
                      lyrics, flags=re.IGNORECASE)[-1]
    
    return lyrics

In [361]:
def zodziu_valymas(zodziai):
    isvalyti = re.sub(r'\[.*?\]', '', zodziai, flags=re.DOTALL)
    
    isvalyti = re.sub(r'\(.*?\)', '', isvalyti)

    isvalyti = re.sub(r'(?i)\b([oaue]+h+|ah+|uh+|yeah|ayy|na|woah|hey|doo|yo)\b',
                      ' ', isvalyti)

    isvalyti = re.sub(r'\b([a-zA-Z])\1{2,}\b', ' ', isvalyti)
    
    isvalyti = re.sub(r'\d+', ' ', isvalyti)

    isvalyti = re.sub(r'\n+', ' ', isvalyti)
    isvalyti = re.sub(r'\s+', ' ', isvalyti)

    return isvalyti

In [362]:
stop_words_gensim = list(STOPWORDS)

def zodziu_apdorojimas(zodziai):
    isvalyti = zodziu_valymas(zodziai)
    doc = nlp(isvalyti)
    
    tokens = [
        token.lemma_.lower()
        for token in doc
        if token.lemma_.lower() not in stop_words_gensim
    ]
    
    return tokens

In [ ]:
zanru_doc_su_pried = []

for zanras, atlikejai in dainos.items():
    for atlikejas, dainu_sarasas in atlikejai.items():
        for pavadinimas in dainu_sarasas:
            zodziai = gauti_zodzius(pavadinimas, atlikejas)
            if zodziai:
                tokens = zodziu_apdorojimas(zodziai)
                zanru_doc_su_pried.append((zanras, tokens))
                print(f"{pavadinimas} [{zanras}]")

In [ ]:
pop_doc_su_pried = [" ".join(tokens) for zanras, tokens in zanru_doc_su_pried 
                    if zanras == "pop"]
rap_doc_su_pried = [" ".join(tokens) for zanras, tokens in zanru_doc_su_pried 
                    if zanras == "rap"]
rock_doc_su_pried = [" ".join(tokens) for zanras, tokens in zanru_doc_su_pried 
                     if zanras == "rock"]

In [ ]:
tfidf_vectorizer_pop = TfidfVectorizer()
X_tfidf_pop = tfidf_vectorizer_pop.fit_transform(pop_doc_su_pried)

tfidf_vectorizer_rap = TfidfVectorizer()
X_tfidf_rap = tfidf_vectorizer_rap.fit_transform(rap_doc_su_pried)

tfidf_vectorizer_rock = TfidfVectorizer()
X_tfidf_rock = tfidf_vectorizer_rock.fit_transform(rock_doc_su_pried)

In [ ]:
lsa_pop = TruncatedSVD(n_components=3, random_state=42).fit(X_tfidf_pop)
lsa_rap = TruncatedSVD(n_components=3, random_state=42).fit(X_tfidf_rap)
lsa_rock = TruncatedSVD(n_components=3, random_state=42).fit(X_tfidf_rock)

In [ ]:
nmf_pop = NMF(n_components=3, random_state=42).fit(X_tfidf_pop)
nmf_rap = NMF(n_components=3, random_state=42).fit(X_tfidf_rap)
nmf_rock = NMF(n_components=3, random_state=42).fit(X_tfidf_rock)

In [ ]:
print("\n------------------- LSA su pried ------------------------------------")
for zanras, modelis, vectorizer in [("pop", lsa_pop, tfidf_vectorizer_pop),
                                 ("rap", lsa_rap, tfidf_vectorizer_rap),
                                 ("rock", lsa_rock, tfidf_vectorizer_rock)]:
    print(f"\n--- {zanras} ---")
    for i, topic in enumerate(modelis.components_):
        top_zodziai = [vectorizer.get_feature_names_out()[j] 
                     for j in topic.argsort()[:-10-1:-1]]
        print(f"Tema {i}: {', '.join(top_zodziai)}")

In [ ]:
print("\n------------------- NMF su pried ------------------------------------")
for zanras, modelis, vectorizer in [("pop", nmf_pop, tfidf_vectorizer_pop),
                                 ("rap", nmf_rap, tfidf_vectorizer_rap),
                                 ("rock", nmf_rock, tfidf_vectorizer_rock)]:
    print(f"\n--- {zanras} ---")
    for i, topic in enumerate(modelis.components_):
        top_zodziai = [vectorizer.get_feature_names_out()[j] 
                     for j in topic.argsort()[:-10-1:-1]]
        print(f"Tema {i}: {', '.join(top_zodziai)}")

In [363]:
def palikti_tik_pirma_priedaini(zodziai):
    eilutes = zodziai.split("\n")
    rezultatas = []

    CHORUS = re.compile(r"^\[chorus.*\]$", re.IGNORECASE)
    pirmas_priedainis_issaugotas = False
    skip_mode = False

    for eil in eilutes:
        eil_strip = eil.strip()
        if CHORUS.match(eil_strip):
            if not pirmas_priedainis_issaugotas:
                pirmas_priedainis_issaugotas = True
                skip_mode = False
                rezultatas.append(eil_strip)
            else:
                skip_mode = True
            continue

        if skip_mode:
            if eil_strip.startswith("[") and len(eil_strip) > 2:
                skip_mode = False
                rezultatas.append(eil_strip)
            continue

        rezultatas.append(eil_strip)

    return "\n".join(rezultatas)

In [366]:
zanru_doc_be_pried = []

for zanras, atlikejai in dainos.items():
    for atlikejas, dainu_sarasas in atlikejai.items():
        for pavadinimas in dainu_sarasas:
            zodziai = gauti_zodzius(pavadinimas, atlikejas)
            if zodziai:
                zodziai = palikti_tik_pirma_priedaini(zodziai)
                tokens = zodziu_apdorojimas(zodziai)
                zanru_doc_be_pried.append((zanras, tokens))
                print(f"{pavadinimas} [{zanras}]")

Royals [pop]
Team [pop]
Ribs [pop]
Tennis Court [pop]
Green Light [pop]
Liability [pop]
Homemade Dynamite [pop]
Solar Power [pop]
Stoned at the Nail Salon [pop]
The Louvre [pop]
Love Story [pop]
Blank Space [pop]
All Too Well [pop]
You Belong With Me [pop]
Cardigan [pop]
Willow [pop]
Wildest Dreams [pop]
Style [pop]
Anti-Hero [pop]
Cruel Summer [pop]
Bad Guy [pop]
Bury a Friend [pop]
When the Party’s Over [pop]
Ocean Eyes [pop]
Everything I Wanted [pop]
Bellyache [pop]
Happier Than Ever [pop]
You Should See Me in a Crown [pop]
Therefore I Am [pop]
Lovely [pop]
Hello [pop]
Someone Like You [pop]
Rolling in the Deep [pop]
Set Fire to the Rain [pop]
Easy On Me [pop]
Skyfall [pop]
When We Were Young [pop]
Send My Love [pop]
Make You Feel My Love [pop]
Turning Tables [pop]
Chandelier [pop]
Elastic Heart [pop]
Cheap Thrills [pop]
Alive [pop]
The Greatest [pop]
Unstoppable [pop]
Bird Set Free [pop]
Big Girls Cry [pop]
Breathe Me [pop]
Soon We'll Be Found [pop]
In Da Club [rap]
Candy Shop [rap

In [367]:
pop_doc_be_pried = [" ".join(tokens) for zanras, tokens in zanru_doc_be_pried 
                    if zanras == "pop"]
rap_doc_be_pried = [" ".join(tokens) for zanras, tokens in zanru_doc_be_pried 
                    if zanras == "rap"]
rock_doc_be_pried = [" ".join(tokens) for zanras, tokens in zanru_doc_be_pried 
                     if zanras == "rock"]

In [ ]:
lengths = {
    "pop su priedainiais": [len(doc.split()) for doc in pop_doc_su_pried],
    "rap su priedainiais": [len(doc.split()) for doc in rap_doc_su_pried],
    "rock su priedainiais": [len(doc.split()) for doc in rock_doc_su_pried],
    "pop be priedainių": [len(doc.split()) for doc in pop_doc_be_pried],
    "rap be priedainių": [len(doc.split()) for doc in rap_doc_be_pried],
    "rock be priedainių": [len(doc.split()) for doc in rock_doc_be_pried]
}

categories = list(lengths.keys())
means = [sum(vals)/len(vals) for vals in lengths.values()]

colors = [
    "#4C72B0", "#55A868",
    "#C44E52", "#4C72B0",
    "#55A868", "#C44E52"
]

plt.figure(figsize=(10,6))
plt.bar(categories, means, color=colors)

plt.xticks(rotation=45, ha="right")
plt.ylabel("Vidutinis žodžių skaičius")
plt.title("Dainų žodžių kiekio palyginimas su ir be priedainių")

plt.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [369]:
tfidf_vectorizer_pop_be = TfidfVectorizer()
X_tfidf_pop_be = tfidf_vectorizer_pop_be.fit_transform(pop_doc_be_pried)

tfidf_vectorizer_rap_be = TfidfVectorizer()
X_tfidf_rap_be = tfidf_vectorizer_rap_be.fit_transform(rap_doc_be_pried)

tfidf_vectorizer_rock_be = TfidfVectorizer()
X_tfidf_rock_be = tfidf_vectorizer_rock_be.fit_transform(rock_doc_be_pried)

In [371]:
lsa_pop_be = TruncatedSVD(n_components=3, random_state=42).fit(X_tfidf_pop_be)
lsa_rap_be = TruncatedSVD(n_components=3, random_state=42).fit(X_tfidf_rap_be)
lsa_rock_be = TruncatedSVD(n_components=3, random_state=42).fit(X_tfidf_rock_be)

In [373]:
nmf_pop_be = NMF(n_components=3, random_state=42).fit(X_tfidf_pop_be)
nmf_rap_be = NMF(n_components=3, random_state=42).fit(X_tfidf_rap_be)
nmf_rock_be = NMF(n_components=3, random_state=42).fit(X_tfidf_rock_be)

In [375]:
print("\n------------------- LSA be pried ------------------------------------")
for zanras, modelis, vectorizer in [("pop", lsa_pop_be, tfidf_vectorizer_pop_be),
                                 ("rap", lsa_rap_be, tfidf_vectorizer_rap_be),
                                 ("rock", lsa_rock_be, tfidf_vectorizer_rock_be)]:
    print(f"\n--- {zanras} ---")
    for i, topic in enumerate(modelis.components_):
        top_zodziai = [vectorizer.get_feature_names_out()[j] 
                     for j in topic.argsort()[:-10-1:-1]]
        print(f"Tema {i}: {', '.join(top_zodziai)}")


------------------- LSA be pried ------------------------------------

--- pop ---
Tema 0: like, know, let, want, cause, time, love, feel, think, wanna
Tema 1: alive, breathe, great, mind, hello, stamina, fight, place, turn, heart
Tema 2: mind, wanna, hello, thought, welcome, want, talk, year, home, feel

--- rap ---
Tema 0: nigga, know, like, love, tell, time, feel, want, niggas, think
Tema 1: nigga, niggas, bitch, lil, sing, club, fuck, shit, song, money
Tema 2: funk, til, humble, roof, sit, leg, spot, let, shop, la

--- rock ---
Tema 0: know, like, away, love, thing, let, time, come, night, mind
Tema 1: away, hold, far, right, fade, let, control, arm, want, run
Tema 2: close, place, heaven, let, ma, bring, think, ask, human, try


In [377]:
print("\n------------------- NMF be pried ------------------------------------")
for zanras, modelis, vectorizer in [("pop", nmf_pop_be, tfidf_vectorizer_pop_be),
                                 ("rap", nmf_rap_be, tfidf_vectorizer_rap_be),
                                 ("rock", nmf_rock_be, tfidf_vectorizer_rock_be)]:
    print(f"\n--- {zanras} ---")
    for i, topic in enumerate(modelis.components_):
        top_zodziai = [vectorizer.get_feature_names_out()[j] 
                     for j in topic.argsort()[:-10-1:-1]]
        print(f"Tema {i}: {', '.join(top_zodziai)}")


------------------- NMF be pried ------------------------------------

--- pop ---
Tema 0: like, know, let, cause, want, think, come, love, time, wake
Tema 1: breathe, alive, lose, turn, table, soon, great, hurt, today, fight
Tema 2: mind, hello, feel, wanna, thought, welcome, anymore, year, home, outside

--- rap ---
Tema 0: know, like, time, feel, think, need, lie, tell, leave, want
Tema 1: nigga, niggas, love, know, like, bitch, shit, sing, fuck, come
Tema 2: funk, til, humble, let, shop, spot, roof, sit, hypnotize, candy

--- rock ---
Tema 0: thing, night, know, like, love, mind, baby, way, time, day
Tema 1: away, hold, let, soul, want, run, right, control, far, arm
Tema 2: close, place, heaven, think, like, bring, let, look, know, cause
